In [4]:
import scipy
from scipy import stats 
import numpy as np
import pandas as pd
import sklearn
from sklearn.preprocessing import MinMaxScaler
from sklearn.preprocessing import RobustScaler



In [5]:
print(f"scipy   : {scipy.__version__}")
print(f"numpy   : {np.__version__}")
print(f"pandas  : {pd.__version__}")
print(f"sklearn : {sklearn.__version__}")

scipy   : 1.15.3
numpy   : 2.2.6
pandas  : 2.3.3
sklearn : 1.7.2


In [ ]:
stats.shapiro()

## Задание 27

Известно, что ежедневное количество посетителей интернет-магазина может меняться от **10** до **35**. Известно, что 25 апреля интернет-магазин посетило **17** уникальных пользователей.

Нормализуйте количество посетителей от 25 апреля с помощью **MinMaxScaler**. Округлите результат до одного знака после точки-разделителя.

In [6]:
visitors_min, visitors_max = 10, 35  # известный диапазон
visitors = 17

# по формуле
visitors_scaled = (visitors - visitors_min) / (visitors_max - visitors_min)

# минмакс скейлер
data = np.array([[visitors]])
scaler = MinMaxScaler()
scaler.fit(np.array([[visitors_min], [visitors_max]]))  # обучаем на границах диапазона

print(f"Формула: {round(visitors_scaled, 1)}")
print(f"Скейлер: {scaler.transform(data)[0][0]}")

Формула: 0.3
Скейлер: 0.28


## Задание 28

В таблице ниже представлены ежедневные показатели количества обращений в службу поддержки на сайте за пять дней:

| № дня	| Количество обращений |
|-------|----------------------|
| 1	    | 30 |
| 2	    | 40 |
| 3	    | 35 |
| 4	    | 57 |
| 5	    | 15 |

Нормализуйте данные с помощью RobustScaler. В качестве ответа запишите нормализованное количество обращений за третий день. Округлите результат до одного знака после точки-разделителя.

In [9]:
data = np.array([30, 40, 35, 57, 15], dtype=float).reshape(-1, 1)

# вручную
q1, median, q3 = np.percentile(data, [25, 50, 75])
scaled_manual = (35 - median) / (q3 - q1)
print(q1, median, q3)

# Способ 1: sklearn
scaler = RobustScaler()
scaled_robust = scaler.fit_transform(data)
print(scaled_robust.flatten())


print(f"Формула: {round(scaled_manual, 1)}")
print(f"Скейлер: {round(scaled_robust[2][0], 1)}")


30.0 35.0 40.0
[-0.5  0.5  0.   2.2 -2. ]
Формула: 0.0
Скейлер: 0.0


## Тестирование новой версии сайта

Одно из самых важных и распространённых применений статистических тестов в бизнесе — это A/B-тестирование.

> &nbsp;  
> **A/B-тестирование** — это метод проверки эффективности изменений в продукте, который заключается в сравнении текущей версии продукта (версии А) с его изменённой версией (версией B) на основании данных, полученных до внедрения обновления в продукт и после него.  
> &nbsp;  

Чтобы понять, являются ли результаты A/B-теста статистически значимыми, как раз используются статистические тесты.

Мы будем подробнее говорить об A/B-тестировании в блоке «Машинное обучение в бизнесе», будем учиться правильно его проводить, а также интерпретировать результаты. Однако уже сейчас, зная основные статистические тесты, мы сможем решить простейшую задачу.


<div style="border: 1px solid #888; padding: 10px;">Компания по продаже бытовой техники представила новую версию сайта с рекомендациями товаров. Предполагается, что новый сайт будет стимулировать посетителей оставаться на нём в течение более длительного времени. Для проведения теста компания перенаправляет часть трафика на обновлённую версию сайта.</div>

Результат с дневным трафиком хранится в файле new-site.

<div style="border: 1px solid #888; padding: 10px;">
<i>Примечание. Разделитель в файле — символ табуляции <b>'\t'</b></i>.
</div>

Каждая строка показывает время пребывания посетителей на сайте. Если посетитель пользовался исходным дизайном, то считается, что он относился к контрольной группе А; если обновлённым — к тестовой группе B.

Необходимо провести небольшое исследование и определить, повлияло ли внедрение рекомендательной системы на длительность сессии пользователей на сайте.

Вам предстоит произвести статический анализ результатов A/B-теста и проверить предложенную бизнес-гипотезу. Итак, приступим!

### Подготовка

In [14]:
DATA_FILE_NAME = 'data/new-site.zip'

original_df = pd.read_csv(DATA_FILE_NAME, sep='\t')

display(original_df.head())
display(original_df.columns.tolist())
original_df.info()

,site,dwell-time
0,0,2
1,1,48
2,0,15
3,1,16
4,0,38


['site', 'dwell-time']

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 300 entries, 0 to 299
Data columns (total 2 columns):
 #   Column      Non-Null Count  Dtype
---  ------      --------------  -----
 0   site        300 non-null    int64
 1   dwell-time  300 non-null    int64
dtypes: int64(2)
memory usage: 4.8 KB


In [26]:
# Поскольку нигде не указано, какой конкретно дизайн сайта кодируется как А а какой как В
# Рискну предположить, что А == 0, а В == 1
# Если первое задание сфейлится, значит я ошибся
SITE_COLUMN=  'site'
TIME_COLUMN = 'dwell-time'

A = 0
B = 1

ALPHA_LEVEL = .05

### Задание 34

Чему равно суммарное время пребывания на сайте в группах А и В? Ответ вводите **без пробелов**.

In [22]:
group_a: pd.DataFrame = original_df[original_df[SITE_COLUMN] == A][TIME_COLUMN]
group_b: pd.DataFrame = original_df[original_df[SITE_COLUMN] == B][TIME_COLUMN]

print('Группа A:', group_a.sum())
print('Группа B:', group_b.sum())

Группа A: 16898
Группа B: 10032


Задание зачтено. Оставляем А == 0, В == 1

### Задание 35

Чему равно среднее время пребывание на сайте в группах? Ответ округлите до **целого числа**.

In [23]:
print('Группа A:', round(group_a.mean()))
print('Группа B:', round(group_b.mean()))

Группа A: 91
Группа B: 88


### Задание 36

Используя информацию по эксперименту с новым сайтом, мы хотим проверить данные на нормальность с помощью критерия Шапиро — Уилка.

Как будут звучать нулевая и альтернативная гипотезы для каждого из тестов?

### Задание 37

Каковы результаты статистического теста на нормальность при уровне значимости a == .05 ?

In [25]:
print('Группа A:', stats.shapiro(group_a))
print('Группа B:', stats.shapiro(group_b))

Группа A: ShapiroResult(statistic=np.float64(0.731232632199737), pvalue=np.float64(4.499873552750829e-17))
Группа B: ShapiroResult(statistic=np.float64(0.7305351552861856), pvalue=np.float64(3.5727031185288115e-13))


### Задание 38

Исходя из результатов тестирования на нормальность, какой тест вы примените для проверки гипотезы о том, что время, проводимое пользователями на сайте, совпадает в группах A и B? Известно, что пользователи в группах не пересекаются, то есть выборки являются независимыми.

Логика выбора: выборки независимы, нормальностьи нет. По-этому Манн-Уитни 

### Задание 39

Проведите статистическое тестирование с помощью теста, выбранного в предыдущем задании, при следующих нулевой и альтернативной гипотезах:

Нулевая гипотеза: время пребывания пользователей на сайте в группах А и B одинаково.
Альтернативная гипотеза: время пребывания пользователей на сайте в группах А и B различно.

Вычислите p-value для времени пребывания на сайте в группах А и B. Сравните полученное значение с уровнем значимости .

Какой вывод можно сделать на основании тестирования?

In [27]:
stat, p_value = stats.mannwhitneyu(group_a, group_b, alternative='two-sided')
print('p-value =', p_value)

if p_value < ALPHA_LEVEL:
    print('Отвергаем H0: время пребывания в группах различается.')
else:
    print('Нет оснований отвергнуть H0: время пребывания одинаково.')

p-value = 0.895811197131841
Нет оснований отвергнуть H0: время пребывания одинаково.
